# N160 · Shell文本流与管道

**目标：** 用真实文本工具完成逐行和按列处理。

**先修：** N006, N010, N023。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** stdin/stdout；管道；grep/awk/sed；sort/uniq；正则；locale；退出码。

## 从问题到算法

Shell 管道传递的是文本流，而不是 DataFrame。先固定输入语法：词由空白分隔且区分大小写；词频并列按 C locale 字典序；电话必须整行匹配；转置要求矩形空白分隔表。

四个脚本直接调用真实 `awk`、`sort`、`sed`。词频先用关联数组计数，再按次数逆序和词升序排序；电话加首尾锚点拒绝多余空格；转置先记录 `(行, 列)`；第十行无需读入 Python。

脚本从标准输入读取，便于管道组合。在文件式判题接口中用 `< file.txt` 重定向即可。矩形前提失败返回非零退出码，不能把错误数据当作空结果。

## 最小实现

**本章接口：** `word_frequency.sh`、`valid_phone_numbers.sh`、`transpose_file.sh`、`tenth_line.sh`

In [1]:
import os
import subprocess
import tempfile
from pathlib import Path

SHELL_SOURCES = {'word_frequency.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1\n", 'valid_phone_numbers.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk '/^([0-9][0-9][0-9]-|\\([0-9][0-9][0-9]\\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'\n", 'transpose_file.sh': '#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nawk \'NR==1 { width=NF }\n     NF!=width { bad=1; exit 2 }\n     { for (i=1;i<=NF;i++) cell[NR,i]=$i }\n     END { if (bad) exit 2;\n           for (i=1;i<=width;i++) {\n             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\\n" : " ")\n           }\n     }\'\n', 'tenth_line.sh': "#!/usr/bin/env bash\nset -euo pipefail\nexport LC_ALL=C\nsed -n '10p'\n"}

def run_shell(name, text):
    """Execute the actual script on stdin in a temporary directory."""
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / name
        path.write_text(SHELL_SOURCES[name], encoding='utf-8')
        result = subprocess.run(['bash', str(path)], input=text, text=True,
            capture_output=True, check=True, timeout=10,
            env={**os.environ, 'LC_ALL':'C'})
        return result.stdout

## 正确性、前提与复杂度

计数不变量是已读前缀中每个词的出现次数。转置的第 i 个输出行，按原行序列出全部 `cell[j,i]`，因此与原矩阵第 i 列一一对应。正则锚点保证是完整合法电话，而不是字符串中存在合法片段。

**代价：** 词频 O(L+u log u)，L 为文本长度、u 为不同词数。电话与第十行线性扫描；转置存储全部 r×c 个字段，时间、空间均 O(rc)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
for name in SHELL_SOURCES:
    display(Code(SHELL_SOURCES[name], language='bash'))
text = 'the day is sunny the the\nthe sunny is is\n'
show_table(['stage','text','lines'], [
 ['input',repr(text),len(text.splitlines())],
 ['frequency',repr(run_shell('word_frequency.sh',text)),len(run_shell('word_frequency.sh',text).splitlines())],
 ['transpose',repr(run_shell('transpose_file.sh','name age\nalice 21\nbob 23')),2]])

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '{ for (i=1;i<=NF;i++) count[$i]++ } END { for (word in count) print word, count[word] }' | sort -k2,2nr -k1,1

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk '/^([0-9][0-9][0-9]-|\([0-9][0-9][0-9]\) )[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]$/ { print }'

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
awk 'NR==1 { width=NF }
     NF!=width { bad=1; exit 2 }
     { for (i=1;i<=NF;i++) cell[NR,i]=$i }
     END { if (bad) exit 2;
           for (i=1;i<=width;i++) {
             for (j=1;j<=NR;j++) printf "%s%s", cell[j,i], (j==NR ? "\n" : " ")
           }
     }'

#!/usr/bin/env bash
set -euo pipefail
export LC_ALL=C
sed -n '10p'

stage,text,lines
input,'the day is sunny the the\nthe sunny is is\n',2
frequency,'the 4\nis 3\nsunny 2\nday 1\n',4
transpose,'name alice bob\nage 21 23\n',2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert run_shell('word_frequency.sh','') == ''
assert run_shell('word_frequency.sh','b a b a c') == 'a 2\nb 2\nc 1\n'
phones = '987-123-4567\n(123) 456-7890\n123 456 7890\n987-123-4567 \n(12) 345-6789\n000-000-0000'
assert run_shell('valid_phone_numbers.sh',phones) == '987-123-4567\n(123) 456-7890\n000-000-0000\n'
assert run_shell('valid_phone_numbers.sh','bad') == ''
assert run_shell('transpose_file.sh','name age\nalice 21\nbob 23') == 'name alice bob\nage 21 23\n'
assert run_shell('transpose_file.sh','') == ''
assert run_shell('transpose_file.sh','only') == 'only\n'
try:
    run_shell('transpose_file.sh','a b\nc')
except subprocess.CalledProcessError as error:
    assert error.returncode == 2
else:
    raise AssertionError('Ragged table was accepted')
assert run_shell('tenth_line.sh','1\n2') == ''
lines = '\n'.join(map(str,range(1,12)))
assert run_shell('tenth_line.sh',lines).strip() == '10'
assert run_shell('tenth_line.sh','').strip() == ''
print('N160: 所有本章断言通过')

N160: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 处理不满10行文件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明sort和locale如何影响排序，不用Python重写题解。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 192. Word Frequency（canonical）
- 193. Valid Phone Numbers（canonical）
- 194. Transpose File（canonical）
- 195. Tenth Line（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。